In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Activation, ReLU
from tensorflow.keras.layers import BatchNormalization, Conv2DTranspose, Concatenate
from tensorflow.keras.models import Model, Sequential

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## import data 

In [ ]:
import os
import numpy as np
input_dir = "/content/drive/MyDrive/SI/data_2023"
target_dir = "/content/drive/MyDrive/SI/mask_via_2023"
img_size = (160, 160)
num_classes = 2
batch_size = 8

input_img_paths = sorted(
    [
        os.path.join(input_dir, fname)
        for fname in os.listdir(input_dir)
        if fname.endswith(".png")
    ]
)
target_img_paths = sorted(
    [
        os.path.join(target_dir, fname)
        for fname in os.listdir(target_dir)
        if fname.endswith(".png") and not fname.startswith(".")
    ]
)

print("Number of samples:", len(input_img_paths))
for input_path, target_path in zip(input_img_paths[:10], target_img_paths[:10]):
    print(input_path, "|", target_path)

In [ ]:
from tensorflow import keras
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array

class SegmentationDataset(keras.utils.Sequence):
    """Keras Sequence for binary segmentation tasks."""

    def __init__(self, batch_size, img_size, input_img_paths, target_img_paths,**kwargs):
        super().__init__(**kwargs)
        self.batch_size = batch_size
        self.img_size = img_size
        self.input_img_paths = input_img_paths
        self.target_img_paths = target_img_paths

    def __len__(self):
        return len(self.target_img_paths) // self.batch_size

    def __getitem__(self, idx):
        i = idx * self.batch_size
        batch_input_img_paths = self.input_img_paths[i: i + self.batch_size]
        batch_target_img_paths = self.target_img_paths[i: i + self.batch_size]

        # Inputs
        x = np.zeros((self.batch_size,) + self.img_size + (3,), dtype="float32")
        for j, path in enumerate(batch_input_img_paths):
            img = load_img(path, target_size=self.img_size)
            img_array = img_to_array(img) / 255.0  # Normalize to [0, 1]
            x[j] = img_array

        # Targets
        y = np.zeros((self.batch_size,) + self.img_size + (1,), dtype="float32")
        for j, path in enumerate(batch_target_img_paths):
            img = load_img(path, target_size=self.img_size, color_mode="grayscale")
            img_array = img_to_array(img) / 255.0  # Normalize 255 → 1
            y[j] = img_array  # shape (H, W, 1), values 0.0 or 1.0

        return x, y


In [ ]:
import random
import os

# Set split ratios
val_ratio = 0.1
test_ratio = 0.1  # 10% for test, adjust as needed

# Shuffle with fixed seed for reproducibility
random.seed(1337)
combined = list(zip(input_img_paths, target_img_paths))
random.shuffle(combined)
input_img_paths, target_img_paths = zip(*combined)

# Calculate sizes
total_samples = len(input_img_paths)
val_samples = int(total_samples * val_ratio)
test_samples = int(total_samples * test_ratio)

# Split into train / val / test
train_input_img_paths = input_img_paths[:-(val_samples + test_samples)]
train_target_img_paths = target_img_paths[:-(val_samples + test_samples)]

val_input_img_paths = input_img_paths[-(val_samples + test_samples):-test_samples]
val_target_img_paths = target_img_paths[-(val_samples + test_samples):-test_samples]

test_input_img_paths = input_img_paths[-test_samples:]
test_target_img_paths = target_img_paths[-test_samples:]


print(f"Train samples: {len(train_input_img_paths)}")
print(f"Val samples:   {len(val_input_img_paths)}")
print(f"Test samples:  {len(test_input_img_paths)}")

# Instantiate data generators
train_gen = SegmentationDataset(batch_size, img_size, train_input_img_paths, train_target_img_paths)
val_gen = SegmentationDataset(batch_size, img_size, val_input_img_paths, val_target_img_paths)
test_gen = SegmentationDataset(batch_size, img_size, test_input_img_paths, test_target_img_paths)


## UNet architecture 

In [ ]:
def convolution_operation(entered_input, filters=64):
    # Taking first input and implementing the first conv block
    conv1 = Conv2D(filters, kernel_size = (3,3), padding = "same")(entered_input)
    batch_norm1 = BatchNormalization()(conv1)
    act1 = ReLU()(batch_norm1)

    # Taking first input and implementing the second conv block
    conv2 = Conv2D(filters, kernel_size = (3,3), padding = "same")(act1)
    batch_norm2 = BatchNormalization()(conv2)
    act2 = ReLU()(batch_norm2)

    return act2

def encoder(entered_input, filters=64):
    # Collect the start and end of each sub-block for normal pass and skip connections
    enc1 = convolution_operation(entered_input, filters)
    MaxPool1 = MaxPooling2D(strides = (2,2))(enc1)
    return enc1, MaxPool1

def decoder(entered_input, skip, filters=64):
    # Upsampling and concatenating the essential features
    Upsample = Conv2DTranspose(filters, (2, 2), strides=2, padding="same")(entered_input)
    Connect_Skip = Concatenate()([Upsample, skip])
    out = convolution_operation(Connect_Skip, filters)
    return out

def U_Net(Image_Size):
    # Take the image size and shape
    input1 = Input(Image_Size)

    # Construct the encoder blocks
    skip1, encoder_1 = encoder(input1, 64)
    skip2, encoder_2 = encoder(encoder_1, 64*2)
    skip3, encoder_3 = encoder(encoder_2, 64*4)
    skip4, encoder_4 = encoder(encoder_3, 64*8)

    # Preparing the next block
    conv_block = convolution_operation(encoder_4, 64*16)

    # Construct the decoder blocks
    decoder_1 = decoder(conv_block, skip4, 64*8)
    decoder_2 = decoder(decoder_1, skip3, 64*4)
    decoder_3 = decoder(decoder_2, skip2, 64*2)
    decoder_4 = decoder(decoder_3, skip1, 64)

    out = Conv2D(1, 1, padding="same", activation="sigmoid")(decoder_4)

    model = Model(input1, out)
    return model

input_shape = (160, 160, 3)
model = U_Net(input_shape)
model.summary()

## train model

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import MeanIoU
from tensorflow.keras.losses import binary_crossentropy
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
import tensorflow.keras.backend as K
import os

# Dice Coefficient implementation
def dice_coef(y_true, y_pred, smooth=1):
    intersection = K.sum(y_true * y_pred, axis=[1,2,3])
    union = K.sum(y_true, axis=[1,2,3]) + K.sum(y_pred, axis=[1,2,3])
    return K.mean((2. * intersection + smooth)/(union + smooth), axis=0)

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

# Combined loss (binary crossentropy + dice)
def combined_loss(y_true, y_pred):
    return binary_crossentropy(y_true, y_pred) + dice_loss(y_true, y_pred)

# Model compilation
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=combined_loss,  # You can also use just 'binary_crossentropy' here
    metrics=['accuracy', dice_coef, MeanIoU(num_classes=2)]
)

# Callbacks setup
os.makedirs('/content/drive/MyDrive/SI/callbacks', exist_ok=True)

callbacks = [
    ModelCheckpoint(
        '/content/drive/MyDrive/SI/callbacks/model_2023_only.keras',
        monitor='val_dice_coef',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath='/content/drive/MyDrive/SI/callbacks/model_2023_last_model.keras',
        save_best_only=False,  # Saves the final model, even if not the best
        verbose=1
    ),

    EarlyStopping(
        monitor='val_dice_coef',
        patience=20,
        mode='max',
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.1,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    CSVLogger('/content/drive/MyDrive/SI/callbacks/training_log.csv')
]


In [ ]:
# Train the model, doing validation at the end of each epoch.
epochs = 30
model.fit(train_gen, epochs=epochs, validation_data=val_gen, callbacks=callbacks)